# Khôi phục model Drive và học tiếp — 2×T4
Gắn Dataset train, bật Internet và T4×2. Dán link Drive khi cell hỏi; URL không được lưu trong code public. Không train lại từ Hugging Face base nếu Drive chứa full model đã fine-tune. Đây là warm-start weights, không phải resume optimizer của runtime đã mất. Đo lại WER trước khi học. Lưu Output/tải model và checkpoint ra ngoài trước khi kết thúc runtime; checkpoint trong /kaggle/working không phải bản sao lưu bền vững.

In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "accelerate>=1.6,<2" soundfile scipy numpy gdown

In [ ]:
from pathlib import Path
import os, sys, subprocess
TRAIN_JSON = Path('/kaggle/input/datasets/n107hoangtuong/zalo-competion/train+public_test/train/train.jsonl')
AUDIO_ROOT = TRAIN_JSON.parent
WORK = Path('/kaggle/working/lifemate-asr')
PUBLIC_REPO = 'https://github.com/tuong1901/zac2026-lifemate.git'
PUBLIC_BRANCH = 'feat-eval-LLM'
if not (WORK / '.git').is_dir():
    assert not WORK.exists() or not any(WORK.iterdir()), 'Choose an empty WORK directory for git clone'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PUBLIC_BRANCH,
                    PUBLIC_REPO, str(WORK)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(WORK), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote == PUBLIC_REPO, f'Unexpected origin: {remote}'
    branch = subprocess.check_output(['git', '-C', str(WORK), 'branch', '--show-current'], text=True).strip()
    assert branch == PUBLIC_BRANCH, f'Checkout {PUBLIC_BRANCH} before updating'
    changes = subprocess.check_output(['git', '-C', str(WORK), 'status', '--porcelain'], text=True)
    if changes.strip():
        print('Local edits found; keeping them. Commit or stash them manually before pulling updates.')
        print(changes)
    else:
        subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only', 'origin', PUBLIC_BRANCH], check=True)
assert (WORK / 'src/asr/finetune.py').is_file()
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
import torch
assert torch.cuda.device_count() == 2, 'Select T4 x2 and restart session before importing torch'
assert TRAIN_JSON.is_file(), f'Check TRAIN_JSON: {TRAIN_JSON}'
assert (AUDIO_ROOT / 'audio').is_dir(), f'Check AUDIO_ROOT: {AUDIO_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', WORK / 'src/asr')
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Training manifest:', TRAIN_JSON)
# After updating code already imported in this kernel, restart the session before importing again.
# The train/merge subprocesses always read the current files on disk.

In [ ]:
import json, gdown
from src.asr.restore import discover_models, unpack_model_zip
DOWNLOADS = Path('/kaggle/working/drive-model-download')
DOWNLOADS.mkdir(parents=True, exist_ok=True)
if not discover_models(DOWNLOADS):
    url = input('Dán link folder Drive chứa model đã lưu: ').strip()
    assert '/folders/' in url, 'Dùng link folder Drive được chia sẻ cho người có link'
    files = gdown.download_folder(url=url, output=str(DOWNLOADS), quiet=False, use_cookies=False)
    assert files, 'Tải Drive thất bại: kiểm tra quyền chia sẻ/quota'
    del url
    for archive in list(DOWNLOADS.rglob('*.zip')):
        unpack_model_zip(archive, DOWNLOADS / 'unpacked' / archive.stem)
candidates = discover_models(DOWNLOADS)
assert candidates, 'Không tìm thấy full Whisper model/adapter đủ weights; kiểm tra file đã tải'
for index, candidate in enumerate(candidates):
    print(index, candidate)
MODEL_CHOICE = None  # Nếu có nhiều model, nhập chỉ số đúng rồi chạy lại cell.
assert len(candidates) == 1 or MODEL_CHOICE is not None, 'Chọn MODEL_CHOICE; không tự đoán model nào'
selected = candidates[0 if MODEL_CHOICE is None else MODEL_CHOICE]
print('Selected:', selected)

In [ ]:
# If it is an adapter, merge with its ORIGINAL base first. Never substitute another base.
ADAPTER_BASE_OVERRIDE = None
if selected['kind'] == 'adapter':
    RESTORED_MODEL = Path('/kaggle/working/restored-original-model')
    command = [sys.executable, '-m', 'src.asr.finetune', 'merge',
               '--adapter', selected['path'], '--out', str(RESTORED_MODEL)]
    if ADAPTER_BASE_OVERRIDE:
        command += ['--model', ADAPTER_BASE_OVERRIDE]
    subprocess.run(command, check=True)
else:
    RESTORED_MODEL = Path(selected['path'])
assert (RESTORED_MODEL / 'preprocessor_config.json').is_file(), 'Cần tải đủ processor/tokenizer gốc cùng model'
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
from transformers import WhisperProcessor
WhisperProcessor.from_pretrained(RESTORED_MODEL, local_files_only=True)
print('Dùng weights đã khôi phục làm base học tiếp:', RESTORED_MODEL)

In [ ]:
from src.asr.training_data import collect_clips, split_clips
ROOT = Path('/kaggle/working/asr_restored_experiment')
ROOT.mkdir(parents=True, exist_ok=True)
saved_splits = list(DOWNLOADS.rglob('split_manifest.json'))
assert len(saved_splits) <= 1, 'Có nhiều split cũ; chọn đúng một file trước khi tiếp tục'
SPLIT = ROOT / 'source_split.json'
if saved_splits:
    manifest = json.loads(saved_splits[0].read_text())
    for row in manifest['train'] + manifest['validation']:
        assert Path(row['audio']).is_file(), f'Sửa đường dẫn audio trong split cũ: {row["audio"]}'
    print('Dùng lại split đã lưu:', saved_splits[0])
else:
    train_rows, val_rows = split_clips(collect_clips(TRAIN_JSON, AUDIO_ROOT), 0.1, 42)
    manifest = {'train': train_rows, 'validation': val_rows, 'seed': 42, 'val_ratio': 0.1,
                'reconstructed': True}
    print('Tái tạo split bằng seed42; chưa xác minh trùng tuyệt đối split bị mất.')
SPLIT.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Train:', len(manifest['train']), 'validation:', len(manifest['validation']))
# Establish a NEW measured baseline on this manifest; don't assume the old 13%/11.81% transfers.

In [ ]:
RUN_TRAIN = False  # Đổi True khi đã kiểm tra đúng model và dữ liệu.
NEW_RUN = ROOT / 'continued_run'
if RUN_TRAIN:
    subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone',
        '--nproc_per_node=2', '-m', 'src.asr.finetune', 'train',
        '--model', str(RESTORED_MODEL), '--train-json', str(TRAIN_JSON),
        '--audio-root', str(AUDIO_ROOT), '--split-manifest', str(SPLIT),
        '--out', str(NEW_RUN), '--epochs', '1', '--batch-size', '1',
        '--grad-accum', '16', '--lr', '2e-5', '--rank', '32',
        '--specaug', '0.05', '--speed', '0.1', '--eval-beams', '5', '--eval-ngram', '0',
        '--save-steps', '50', '--seed', '42'], check=True)
else:
    print('Chưa train. Kiểm tra model/split rồi đặt RUN_TRAIN=True.')
# New LoRA on restored FULL weights, fresh optimizer, same validation before/after.
# Local checkpoint every 50 updates + final update. These are NOT persistent backups.

In [ ]:
assert RUN_TRAIN, 'Chạy train thành công trước'
before = json.loads((NEW_RUN / 'baseline_metrics.json').read_text())['baseline_wer']
after = json.loads((NEW_RUN / 'validation_metrics.json').read_text())['eval_wer']
print(f'Trước: {before:.2%}; sau: {after:.2%}')
if after < before:
    FINAL_MODEL = ROOT / 'improved-model'
    subprocess.run([sys.executable, '-m', 'src.asr.finetune', 'merge',
        '--adapter', str(NEW_RUN / 'best_adapter'), '--model', str(RESTORED_MODEL),
        '--out', str(FINAL_MODEL)], check=True)
else:
    FINAL_MODEL = RESTORED_MODEL
    print('Không cải thiện: giữ weights ban đầu, không ghi đè.')
# Save the measured decoding settings even if the original model wins.
from transformers import GenerationConfig
try:
    generation = GenerationConfig.from_pretrained(FINAL_MODEL)
except OSError:
    from transformers import WhisperConfig
    generation = GenerationConfig.from_model_config(WhisperConfig.from_pretrained(FINAL_MODEL))
generation.num_beams = 5
generation.no_repeat_ngram_size = 0
generation.save_pretrained(FINAL_MODEL)

In [ ]:
import shutil
model_archive = shutil.make_archive('/kaggle/working/phowhisper-selected-model', 'zip', FINAL_MODEL)
run_archive = shutil.make_archive('/kaggle/working/phowhisper-training-records', 'zip', NEW_RUN)
print('TẢI HAI FILE NÀY NGAY TRONG OUTPUT:')
print(model_archive)
print(run_archive)
print('Lưu cả notebook Output bằng Save Version nếu cần. /kaggle/working có thể mất khi hết runtime.')
print('Không đẩy weights/dữ liệu lên GitHub public. Model trên Drive cũ vẫn được giữ để khôi phục.')

## Chỉ chạy public_test bằng model Drive và model mới
Đặt RUN_COMPARE=True sau khi có model mới. Mặc định chỉ public_test; hai model chạy lần lượt, mỗi lượt dùng cả 2 GPU. Xuất transcript riêng và bảng đối chiếu, không chạy train/validation, không có WER vì không có nhãn chuẩn. Không dùng transcript public_test để bổ sung nhãn train. Đây chỉ là ASR, không tạo tool calls/state_hash submission.

In [ ]:
RUN_COMPARE = False  # Đổi True để chạy so sánh.
COMPARE_SPLIT = 'public_test'  # Chỉ public_test theo yêu cầu.
TEST_JSON = TRAIN_JSON.parent.parent / 'public_test/public_test.jsonl'
# Nếu đường dẫn khác, Copy path file public_test.jsonl trong Input và sửa TEST_JSON.
DRIVE_MODEL = RESTORED_MODEL  # Weights gốc tải từ Drive, không phải FINAL_MODEL.
assert COMPARE_SPLIT in ('public_test', 'validation', 'train')
if RUN_COMPARE:
    from datetime import datetime, timezone
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    COMPARE_OUT = Path('/kaggle/working/asr_model_comparison') / stamp
    # Even if the new model is worse and FINAL_MODEL kept the old one, merge the
    # actual trained adapter separately so the comparison uses TWO different weights.
    NEW_MODEL_FOR_TEST = ROOT / f'new-model-for-test-{stamp}'
    assert TEST_JSON.is_file(), f'Kiểm tra TEST_JSON: {TEST_JSON}'
    assert (NEW_RUN / 'best_adapter/adapter_config.json').is_file(), 'Cần train xong hoặc recover checkpoint trước'
    subprocess.run([sys.executable, '-m', 'src.asr.finetune', 'merge',
        '--adapter', str(NEW_RUN / 'best_adapter'), '--model', str(DRIVE_MODEL),
        '--out', str(NEW_MODEL_FOR_TEST)], check=True)
    command = [sys.executable, '-m', 'src.asr.compare',
        '--drive-model', str(DRIVE_MODEL), '--new-model', str(NEW_MODEL_FOR_TEST),
        '--test-jsonl', str(TEST_JSON), '--out', str(COMPARE_OUT)]
    if COMPARE_SPLIT == 'validation':
        command += ['--validation-manifest', str(NEW_RUN / 'split_manifest.json')]
    elif COMPARE_SPLIT == 'train':
        command += ['--train-jsonl', str(TRAIN_JSON), '--train-audio-root', str(AUDIO_ROOT)]
    subprocess.run(command, check=True)
    print(json.loads((COMPARE_OUT / 'comparison_report.json').read_text()))
else:
    print('Đặt RUN_COMPARE=True khi đã có cả model Drive và model mới.')

In [ ]:
if RUN_COMPARE:
    import shutil
    result_zip = shutil.make_archive(str(COMPARE_OUT) + "-results", "zip", COMPARE_OUT)
    print("Tải file kết quả trong Output:", result_zip)
    print("Các file *_transcripts.jsonl có id/audio_text; *_comparison.jsonl chứa hai transcript cạnh nhau.")